# exp115_hidden_like_spatial_holdout_from_ppt train

Build a deterministic hidden-like spatial holdout from the official PPT Verification map distribution.

## Contents

1. Setup and configuration
2. Input checks
3. PPT red Verification extraction
4. Holdout generation
5. Metrics and artifacts

## 1. Setup and configuration

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd

from settings import EXPERIMENT_NAME, ExperimentPaths, get_nested, load_config
from hidden_like_spatial_holdout_from_ppt import (
    add_ppt_distances,
    build_holdout_assignments,
    build_well_metadata,
    distribution_report,
    extract_red_points_from_ppt,
    fallback_target_points,
    write_outputs,
)

paths = ExperimentPaths()
paths.require_kaggle_runtime()
paths.ensure_output_dirs()
config = load_config()

print('Experiment:', EXPERIMENT_NAME)
print('Route:', get_nested(config, 'experiment.route'))
print('Parent:', get_nested(config, 'lineage.parent'))
print('Validation:', get_nested(config, 'validation.strategy'))
print('Train data:', paths.train_data_dir)
print('Raw data:', paths.raw_data_dir)
print('Artifacts:', paths.artifacts_dir)

## 2. Input checks

In [ ]:
ppt_path = paths.raw_data_dir / str(get_nested(config, 'data.official_pptx'))
train_files = sorted(paths.train_data_dir.glob('*__horizontal_well.csv'))
typewell_files = sorted(paths.train_data_dir.glob('*__typewell.csv'))

print('Official PPT:', ppt_path, 'exists=', ppt_path.exists())
print('Horizontal wells:', len(train_files))
print('Typewells:', len(typewell_files))
print('Target holdout wells:', get_nested(config, 'audit.target_holdout_wells'))
print('Sample train files:', [path.name for path in train_files[:5]])

if not ppt_path.exists():
    raise FileNotFoundError(ppt_path)
if not train_files:
    raise FileNotFoundError(paths.train_data_dir)

## 3. PPT red Verification extraction

In [ ]:
try:
    target_points, ppt_meta = extract_red_points_from_ppt(config, paths)
except Exception as exc:
    if not bool(get_nested(config, 'ppt.allow_fallback')):
        raise
    target_points = fallback_target_points(config)
    ppt_meta = {
        'status': 'fallback_grid',
        'error': f'{type(exc).__name__}: {exc}',
        'slide_number': int(get_nested(config, 'ppt.slide_number') or 10),
    }

print(json.dumps(ppt_meta, indent=2, sort_keys=True))
display(target_points.head(20))

## 4. Holdout generation

In [ ]:
well_metadata = build_well_metadata(paths, config)
well_metadata = add_ppt_distances(well_metadata, target_points)
assignment, holdout_wells = build_holdout_assignments(config, well_metadata, target_points)
report = distribution_report(assignment, target_points)

print('Well metadata rows:', len(well_metadata))
print('Spatial holdout wells:', int((assignment['verification_like_spatial_role'] == 'valid').sum()))
print('Typewell-purged valid wells:', int((assignment['verification_like_typewell_purged_role'] == 'valid').sum()))
print('Typewell-purged excluded train wells:', int((assignment['verification_like_typewell_purged_role'] == 'purged_train_excluded').sum()))
display(holdout_wells.head(30))

## 5. Metrics and artifacts

In [ ]:
summary = write_outputs(paths, config, target_points, ppt_meta, assignment, holdout_wells, report)
print(json.dumps(summary, indent=2, sort_keys=True))

artifact_paths = {name: Path(path) for name, path in summary['artifacts'].items()}
for name, path in artifact_paths.items():
    print(name, path, path.exists(), path.stat().st_size if path.exists() else None)

display(pd.read_csv(artifact_paths['holdout_wells']).head(40))
display(pd.read_csv(artifact_paths['distribution_report']).head(60))